In [89]:
#implementing parts of GPT Transformer
import torch
import torch.nn as nn
import tiktoken

In [90]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length":256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [91]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = torch.ones(emb_dim)
        self.shift = torch.zeros(emb_dim)

    def forward(self,x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x-mean)/torch.sqrt(var+self.eps)
        return self.scale + norm_x + self.shift

class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers=  nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4*cfg['emb_dim']), 
            nn.GELU(),
            nn.Linear(4*cfg['emb_dim'],cfg['emb_dim'])

        )

    def forward(self,x):
        return self.layers(x)




In [92]:
from attention_mechanism import MultiHeadAttention


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm = LayerNorm(cfg['emb_dim'])
        self.dropout = nn.Dropout(cfg['drop_rate'])

    def forward(self,x):
        #shortcut connection for attention block
        shortcut = x
        x = self.norm(x)
        x = self.att(x)
        x = self.dropout(x)
        x = x + shortcut

        return x
        
    

In [93]:
#testing the config 


x = torch.rand(2,6,768)
model = TransformerBlock(GPT_CONFIG_124M)
output = model(x)

print(output.shape), x.shape

torch.Size([2, 6, 768])


(None, torch.Size([2, 6, 768]))

In [94]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
        self.pos_emb = nn.Embedding(cfg['context_length'], cfg['emb_dim'])
        self.drop_emb = nn.Dropout(cfg['drop_rate'])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg['n_layers']) ]
        )

        self.norm = LayerNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )  #maps emb dim to final vocabsize for probability softmax

    def forward(self, input):
        batch_size, input_len = input.shape
        tok_embeds = self.tok_emb(input)
        pos_embeds = self.pos_emb(torch.arange(input_len))
        x = tok_embeds + pos_embeds
        x= self.drop_emb(x)
        x= self.trf_blocks(x)
        x = self.norm(x)
        logits = self.out_head(x)
        return logits
        

In [95]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

#testing 

model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
out

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


tensor([[[ 0.0079, -1.8683, -0.4511,  ...,  0.4232,  0.8570, -0.2095],
         [ 0.3503, -0.6254, -0.7744,  ...,  0.5987,  0.9490,  0.5553],
         [ 0.7555, -1.0084, -0.3107,  ...,  0.7804,  1.0433, -0.2722],
         [ 1.0342, -1.6983, -0.4850,  ...,  0.6188,  1.5293, -0.4840]],

        [[-0.0301, -1.9470, -0.6228,  ...,  0.5425,  1.2928, -0.1035],
         [-0.0062, -1.5780, -0.6378,  ...,  0.0995,  1.7473,  0.2970],
         [ 0.1927, -1.1788, -1.3371,  ...,  0.6175,  1.7951, -0.4224],
         [ 0.9528, -2.0158, -0.7546,  ...,  0.8469,  1.6734, -0.1963]]],
       grad_fn=<UnsafeViewBackward0>)

In [96]:
#text generation using gpt model


def generate_text_simple(model, ids, max_new_tokens, context_size):


    for _ in range(max_new_tokens):

        ids_cond = ids[:, -context_size:]


        with torch.no_grad():
            logits = model(ids_cond)
        
        logits = logits[:, -1, :]

        probas = torch.softmax(logits, dim=-1)

        ids_next = torch.argmax(probas, dim =-1, keepdim=True)

        ids = torch.cat((ids, ids_next), dim=1)
    
    return ids 

In [97]:
model.eval();

import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
start_context = "Sajal is "


def text_to_ids(text, tokenizer=tokenizer):
    encoded = tokenizer.encode(start_context)
    encoded_tensor = torch.tensor(encoded)
    encoded_tensor = encoded_tensor.unsqueeze(0) # added batch dimension
    return encoded_tensor 

token_ids = text_to_ids(start_context, tokenizer)

def ids_to_text(token_ids, tokenizer=tokenizer):
    decoded = tokenizer.decode(token_ids.squeeze(0).tolist())
    return decoded

ids_to_text(token_ids, tokenizer)


out =  generate_text_simple(
    model = model,
    ids = token_ids,
    max_new_tokens = 3,
    context_size= GPT_CONFIG_124M['context_length']
)

ids_to_text(out)


'Sajal is 107ractionraction'

In [98]:
import os
import requests

filepath="the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(filepath):
    response = requests.get(url)
    text_data = response.text
    with open(filepath, "w") as f:
        f.write(text_data)
else:
    with open(filepath, "r") as f:
        text_data = f.read()

print(text_data[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [99]:
from Building_LLM import create_dataloader_v1


train_ratio = 0.9

split = int(train_ratio * len(text_data))
train_data = text_data[:split]
val_data = text_data[split:]
print(val_data)

train_loader = create_dataloader_v1(
    train_data,
    batch_size = 2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

total_tokens = len(tokenizer.encode(text_data))

if total_tokens * split < GPT_CONFIG_124M['context_length']:
    print('NOT ENOUGH TOKENS< REDUCE CNTEXT LENGTH OR INCREASE TRAINING RATIO')

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")



ue' collapsed like a house of cards. He didn't sneer, you understand, poor Stroud--he just lay there quietly watching, and on his lips, through the gray beard, I seemed to hear the question: 'Are you sure you know where you're coming out?'

"If I could have painted that face, with that question on it, I should have done a great thing. The next greatest thing was to see that I couldn't--and that grace was given me. But, oh, at that minute, Rickham, was there anything on earth I wouldn't have given to have Stroud alive before me, and to hear him say: 'It's not too late--I'll show you how'?

"It _was_ too late--it would have been, even if he'd been alive. I packed up my traps, and went down and told Mrs. Stroud. Of course I didn't tell her _that_--it would have been Greek to her. I simply said I couldn't paint him, that I was too moved. She rather liked the idea--she's so romantic! It was that that made her give me the donkey. But she was terribly upset at not getting the portrait--she di

In [100]:
for x, y in train_loader:
    print(x.shape, y.shape)

print('\nVal loader:')
for x, y in val_loader:
    print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Val loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [105]:
#next implement a utility function to calculate the cross-entropy of a given batch
device = 'cuda' if torch.cuda.is_available() else 'cpu'


def calc_loss_batch(input_batch, target_batch, model, device):
     input_batch, target_batch = input_batch.to(device), target_batch.to(device)
     logits = model(input_batch)
     loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
     return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss= 0
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches= min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break

    return total_loss / num_batches

model.to(device);

In [107]:
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)  
    val_loss = calc_loss_loader(val_loader,model, device)


print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 11.11064232720269
Validation loss: 11.053572654724121
